# 04 — Train: GRU/LSTM direct forecaster

Fits one fixed recurrent candidate for the configured target station and evaluates it once on the sealed test feature artifact. The default is a GRU; change `CELL_TYPE` to `"lstm"` to run the same MVP with an LSTM.

**Inputs:** train-derived and test-derived Stage-3 feature artifacts  
**Outputs:** in-notebook loss curve, prediction preview, and native-unit test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins every knob of the run. The configuration deliberately describes **one** candidate per execution: there is no validation split, no early stopping, no tuning, no model persistence and no experiment tracking. Changing a value below and re-running is the entire experiment loop.

**What the imports provide**

- `torch`, `nn` — the network, its optimiser and its loss.
- `DataLoader`, `TensorDataset` — mini-batching and shuffling of the training sequences.
- `StandardScaler` — fitted twice below, once for predictors and once for targets.
- `matplotlib` — the training-loss curve, which is the only in-run diagnostic available given there is no validation set.
- `mean_absolute_error`, `root_mean_squared_error` — the reported metrics, computed after predictions are converted back to water-level units.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed` | Directory the Stage-3 feature Parquets are read from. Nothing is written back. |
| `CELL_TYPE` | `"gru"` | Which recurrent cell to use. A GRU has two gates (update, reset) and fewer parameters; an LSTM has three gates plus a separate long-term cell state, so it is more expressive and slower. Set it to `"lstm"` and re-run to get the other variant — the validation immediately below rejects any other value, so a typo fails at Setup instead of at the fit. |
| `SEQUENCE_LENGTH` | `168` | Length of the encoder window in hours: one full week of predictor rows ending at (and including) the issue time. It also determines the warm-up cost — the first 167 rows of each split can never be a forecast origin, because there is not enough history in front of them. |
| `HIDDEN_UNITS` | `64` | Width of the recurrent hidden state, i.e. how much the cell can carry forward from one hour to the next. This is the model's main capacity knob. |
| `EPOCHS` | `50` | Full passes over the training sequences. With no validation split and no early stopping, this is a fixed budget rather than a stopping criterion — the loss curve below is what tells you whether it was roughly the right one. |
| `BATCH_SIZE` | `64` | Sequences per gradient update. Larger batches give smoother gradients and better hardware utilisation; smaller ones add regularising noise and update more often per epoch. |
| `LEARNING_RATE` | `0.001` | Adam's step size, and its default. It is held constant — there is no scheduler. |
| `SEED` | `42` | Seeds Python's `random`, NumPy, Torch (and CUDA if present) plus the `DataLoader`'s shuffle generator, so weight initialisation and batch order reproduce. Note this makes the run *repeatable*, not bit-identical across devices: MPS and CUDA kernels are not guaranteed to reduce in the same order. |
| `PREDICTION_PREVIEW_ORIGINS` | `5` | Forecast origins shown in the long-form preview at the end (5 origins x 24 horizons = 120 rows). Display only. |
| `DEVICE` | first available of MPS / CUDA / CPU | Where the tensors live. Chosen automatically, so the notebook runs on an Apple GPU, an NVIDIA GPU or plain CPU without edits. |
| `FEATURE_COLUMNS` | 53 names | The frozen Stage-3 predictor contract, read from `feature_column_names()` so the notebook fails loudly if Stage 3 changes it. Every one of the 53 is fed to the network at every one of the 168 timesteps. |
| `TARGET_COLUMNS` | `target_t_plus_01` … `target_t_plus_24` | The 24 future water levels, emitted in a single forward pass — a *direct* multi-horizon setup with no recursive feedback. The guard below asserts there are exactly `FORECAST_HORIZON_HOURS` of them. |

`seed_everything(seed)` applies the seed to every RNG involved, and the displayed table is the run's record of what was actually configured, including which device it ended up on.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from src.config import FORECAST_HORIZON_HOURS, TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
CELL_TYPE = "gru"
SEQUENCE_LENGTH = 168
HIDDEN_UNITS = 64
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.001
SEED = 42
PREDICTION_PREVIEW_ORIGINS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())

if CELL_TYPE not in {"gru", "lstm"}:
    raise ValueError(f"CELL_TYPE must be 'gru' or 'lstm'; got {CELL_TYPE!r}")
if len(TARGET_COLUMNS) != FORECAST_HORIZON_HOURS:
    raise ValueError("Configured target columns do not match FORECAST_HORIZON_HOURS")

DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


def seed_everything(seed: int) -> None:
    """Seed the fixed MVP training run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
display(
    pd.DataFrame(
        [
            {
                "cell_type": CELL_TYPE,
                "sequence_length_hours": SEQUENCE_LENGTH,
                "hidden_units": HIDDEN_UNITS,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "seed": SEED,
                "device": str(DEVICE),
            }
        ]
    )
)

## Feature-contract and sequence helpers

**`read_feature_artifact(path, *, station_id, artifact_name) -> pd.DataFrame`**

- `path` — the Parquet to load.
- `station_id` / `artifact_name` — used to make any raised error name the split it came from.

Loads one Stage-3 artifact and refuses it unless it satisfies the model contract: every required column present, non-empty, exactly one station, timezone-aware timestamps that are unique, increasing and form a contiguous hourly grid, no null `target_valid`, and all 53 predictors plus 24 targets numeric and free of infinities. A recurrent model slides a fixed-width window over row *positions*, so a silently missing hour would quietly change what "168 hours of history" means — hence the grid check.

**`build_sequences(frame, *, sequence_length, station_id, artifact_name) -> (sequences, targets, origins)`**

- `frame` — one validated artifact.
- `sequence_length` — the encoder window, `SEQUENCE_LENGTH` (168).

Turns rows into supervised samples. A row position qualifies as a forecast origin only when all three hold: the `sequence_length` rows ending there have every predictor finite (checked with a rolling count, so a single missing hour invalidates the whole window), Stage 3 marked that issue time `target_valid`, and its 24 targets are finite. It returns the stacked `(n, 168, 53)` predictor windows, the matching `(n, 24)` targets, and the origin rows themselves so timestamps stay attached to their forecasts. If a split yields no eligible window it raises rather than returning an empty array.

Each split is processed **independently**. That is the leakage boundary in this notebook: the test builder is handed only the sealed test artifact, so a test window can never reach back into training hours even though they are adjacent on the real-world timeline.

**`metric_tables(actual, predictions, *, station_id)`** — one aggregate MAE/RMSE over all `n x 24` values plus the same pair per lead hour, all in water-level units (predictions are inverse-transformed before they get here). RMSE is always at least MAE and is dominated by the worst misses, so a wide gap between them means a few large errors rather than uniformly poor accuracy.

**`prediction_preview(origins, actual, predictions)`** — a long-form table for the first `PREDICTION_PREVIEW_ORIGINS` origins, one row per (origin, horizon) pair with the issue timestamp, the target timestamp, the horizon and both values, so each number can be traced to a specific hour.

In [ ]:
def read_feature_artifact(
    path: Path, *, station_id: str, artifact_name: str
) -> pd.DataFrame:
    """Load one Stage-3 feature artifact and validate its model contract."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing feature artifact for {station_id}: {path}")

    frame = pd.read_parquet(path).copy()
    required = {
        "timestamp",
        "station_id",
        "target_valid",
        *FEATURE_COLUMNS,
        *TARGET_COLUMNS,
    }
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")
    if frame["station_id"].dropna().unique().tolist() != [station_id]:
        raise ValueError(
            f"{artifact_name} artifact must contain only station {station_id!r}"
        )

    timestamps = pd.DatetimeIndex(frame["timestamp"])
    if timestamps.tz is None:
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be timezone-aware"
        )
    if timestamps.has_duplicates or not timestamps.is_monotonic_increasing:
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be unique and increasing"
        )
    expected_grid = pd.date_range(timestamps[0], periods=len(timestamps), freq="h")
    if not timestamps.equals(expected_grid):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be a contiguous hourly grid"
        )
    if frame["target_valid"].isna().any():
        raise ValueError(f"{station_id} {artifact_name} target_valid contains nulls")

    numeric_columns = [*FEATURE_COLUMNS, *TARGET_COLUMNS]
    try:
        numeric_values = frame[numeric_columns].to_numpy(dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(
            f"{station_id} {artifact_name} artifact has non-numeric required values"
        ) from error
    if np.isinf(numeric_values).any():
        raise ValueError(
            f"{station_id} {artifact_name} artifact has non-finite required values"
        )
    valid_targets = frame["target_valid"].eq(True)
    if frame.loc[valid_targets, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return frame


def build_sequences(
    frame: pd.DataFrame, *, sequence_length: int, station_id: str, artifact_name: str
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Build split-local recurrent samples from complete predictor windows."""
    if sequence_length < 1:
        raise ValueError("sequence_length must be at least 1")

    predictors = frame[FEATURE_COLUMNS].to_numpy(dtype=float)
    targets = frame[TARGET_COLUMNS].to_numpy(dtype=float)
    finite_predictor_rows = np.isfinite(predictors).all(axis=1)
    complete_windows = (
        pd.Series(finite_predictor_rows, index=frame.index)
        .rolling(sequence_length, min_periods=sequence_length)
        .sum()
        .eq(sequence_length)
        .to_numpy()
    )
    eligible = (
        complete_windows
        & frame["target_valid"].eq(True).to_numpy()
        & np.isfinite(targets).all(axis=1)
    )
    origin_positions = np.flatnonzero(eligible)
    if not len(origin_positions):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has no eligible {sequence_length}-hour sequences"
        )

    sequences = np.stack(
        [
            predictors[position - sequence_length + 1 : position + 1]
            for position in origin_positions
        ]
    )
    return sequences, targets[origin_positions], frame.iloc[origin_positions].copy()


def metric_tables(
    actual: np.ndarray, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE in water-level units."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(actual.ravel(), predictions.ravel()),
                "rmse": root_mean_squared_error(actual.ravel(), predictions.ravel()),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(
                    actual[:, horizon - 1], predictions[:, horizon - 1]
                ),
                "rmse": root_mean_squared_error(
                    actual[:, horizon - 1], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon


def prediction_preview(
    origins: pd.DataFrame, actual: np.ndarray, predictions: np.ndarray
) -> pd.DataFrame:
    """Return a long-form preview for the first configured number of forecast origins."""
    preview_count = min(PREDICTION_PREVIEW_ORIGINS, len(origins))
    horizon_hours = np.arange(1, FORECAST_HORIZON_HOURS + 1)
    issue_timestamps = origins["timestamp"].iloc[:preview_count].to_numpy()
    return pd.DataFrame(
        {
            "issue_timestamp": np.repeat(issue_timestamps, FORECAST_HORIZON_HOURS),
            "target_timestamp": np.repeat(issue_timestamps, FORECAST_HORIZON_HOURS)
            + pd.to_timedelta(np.tile(horizon_hours, preview_count), unit="h"),
            "horizon_hours": np.tile(horizon_hours, preview_count),
            "actual_water_level": actual[:preview_count].ravel(),
            "predicted_water_level": predictions[:preview_count].ravel(),
        }
    )

## Load split-local samples

Both artifacts are read and converted to sequences, each entirely on its own.

The consequence is stated explicitly by the assertion: the test builder receives only the sealed test artifact, so its first 167 rows cannot become forecast origins even though the training rows immediately preceding them exist on the overall timeline. Those rows are the price of a hard split — borrowing them would leak training observations into the test model's encoder.

The displayed table reports how many usable sequences each split produced, the `(168, 53)` shape of one sample, and the size of the discarded test warm-up.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
train_features = read_feature_artifact(
    train_path, station_id=station_id, artifact_name="train"
)
test_features = read_feature_artifact(
    test_path, station_id=station_id, artifact_name="test"
)

train_sequences, train_targets, train_origins = build_sequences(
    train_features,
    sequence_length=SEQUENCE_LENGTH,
    station_id=station_id,
    artifact_name="train",
)
test_sequences, test_targets, test_origins = build_sequences(
    test_features,
    sequence_length=SEQUENCE_LENGTH,
    station_id=station_id,
    artifact_name="test",
)

assert test_origins.index.min() >= test_features.index[SEQUENCE_LENGTH - 1]
display(
    pd.DataFrame(
        [
            {
                "station_id": station_id,
                "train_sequences": len(train_sequences),
                "test_sequences": len(test_sequences),
                "sequence_shape": tuple(train_sequences.shape[1:]),
                "excluded_test_warmup_rows": SEQUENCE_LENGTH - 1,
            }
        ]
    )
)

## Standardize, fit, and predict

**Scaling.** The predictor scaler is fitted on the training windows reshaped to `(n x 168, 53)` — every predictor value inside a training window, flattened across time — so each of the 53 columns is centred and unit-scaled. Because windows overlap by 167 hours, most rows appear in many windows and are effectively counted many times; for an MVP that is accepted, and it is applied identically to train and test so it does not bias the comparison. The target scaler is fitted on the training target vectors. Test data goes through `transform` only, and predictions are put back through `inverse_transform` before any metric is computed, so every reported number is in water-level units.

Scaling is not optional for a recurrent net: unscaled inputs spanning very different ranges make the early gradients dominated by whichever column happens to be largest, and unscaled targets far from zero force the network to spend its first epochs learning a constant offset.

**The model.** `RecurrentForecaster` is deliberately the smallest thing that can do the job:

| Argument | Value | What it does |
| --- | --- | --- |
| `cell_type` | `CELL_TYPE` | Selects `nn.GRU` or `nn.LSTM`. Everything else is identical between the two variants. |
| `input_size` | `53` | Features per timestep — the full Stage-3 predictor vector. |
| `hidden_size` | `HIDDEN_UNITS` (64) | Width of the hidden state carried across the 168 steps. |
| `num_layers` | `1` | A single recurrent layer. Stacking more is the obvious next capacity increase and is intentionally not done here. |
| `batch_first` | `True` | Tensors are shaped `(batch, time, feature)` instead of PyTorch's default `(time, batch, feature)`, which is what the rest of this cell assumes. |

`forward` runs the sequence, keeps only the final timestep's hidden output (`outputs[:, -1, :]` — the state after the whole week has been read) and maps it through one `nn.Linear(64, 24)` head. All 24 horizons come out of that single linear layer at once, so nothing is fed back autoregressively and errors cannot compound across lead times.

**The training loop.**

- `torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)` — adaptive per-parameter step sizes; the default choice for recurrent nets.
- `nn.MSELoss()` — mean squared error on **scaled** targets, which is why the plotted loss is unitless and only comparable between runs of this notebook.
- `DataLoader(..., shuffle=True, generator=loader_generator)` — shuffles sequences between epochs (each sample is an independent window, so ordering them by time would only correlate consecutive batches), with a seeded generator so the shuffle reproduces.
- Each epoch accumulates `loss.item() * len(batch)` and divides by the sample count, giving a correct mean even when the final batch is short.
- `model.eval()` plus `torch.no_grad()` for inference — disables gradient tracking and puts any train-only layers into evaluation mode. The whole test set is predicted in one pass, then the shape is asserted so a silent broadcasting bug cannot reach the metrics.

In [ ]:
predictor_scaler = StandardScaler()
target_scaler = StandardScaler()

train_sequence_shape = train_sequences.shape
test_sequence_shape = test_sequences.shape
train_predictors = predictor_scaler.fit_transform(
    train_sequences.reshape(-1, len(FEATURE_COLUMNS))
).reshape(train_sequence_shape)
test_predictors = predictor_scaler.transform(
    test_sequences.reshape(-1, len(FEATURE_COLUMNS))
).reshape(test_sequence_shape)
train_targets_scaled = target_scaler.fit_transform(train_targets)


class RecurrentForecaster(nn.Module):
    """One-layer direct multi-horizon GRU or LSTM forecaster."""

    def __init__(self, *, cell_type: str, input_size: int, hidden_units: int) -> None:
        """Build the recurrent encoder and the multi-horizon linear head."""
        super().__init__()
        recurrent_class = nn.GRU if cell_type == "gru" else nn.LSTM
        self.recurrent = recurrent_class(
            input_size=input_size,
            hidden_size=hidden_units,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_units, FORECAST_HORIZON_HOURS)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Predict all forecast horizons from the final recurrent step."""
        outputs, _ = self.recurrent(inputs)
        return self.head(outputs[:, -1, :])


model = RecurrentForecaster(
    cell_type=CELL_TYPE,
    input_size=len(FEATURE_COLUMNS),
    hidden_units=HIDDEN_UNITS,
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_function = nn.MSELoss()

train_dataset = TensorDataset(
    torch.tensor(train_predictors, dtype=torch.float32),
    torch.tensor(train_targets_scaled, dtype=torch.float32),
)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator
)

epoch_losses: list[float] = []
for _ in range(EPOCHS):
    model.train()
    weighted_loss = 0.0
    sample_count = 0
    for batch_predictors, batch_targets in train_loader:
        batch_predictors = batch_predictors.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_function(model(batch_predictors), batch_targets)
        loss.backward()
        optimizer.step()
        weighted_loss += loss.item() * len(batch_predictors)
        sample_count += len(batch_predictors)
    epoch_losses.append(weighted_loss / sample_count)

model.eval()
with torch.no_grad():
    scaled_test_predictions = (
        model(torch.tensor(test_predictors, dtype=torch.float32, device=DEVICE))
        .cpu()
        .numpy()
    )
test_predictions = target_scaler.inverse_transform(scaled_test_predictions)
if test_predictions.shape != (len(test_origins), FORECAST_HORIZON_HOURS):
    raise RuntimeError(
        f"Expected ({len(test_origins)}, {FORECAST_HORIZON_HOURS}) predictions; "
        f"got {test_predictions.shape}"
    )

## Training diagnostic and test evaluation

The loss curve is the only in-run diagnostic this notebook has, since there is no validation set. Read it for shape rather than for level: still falling steeply at epoch 50 suggests `EPOCHS` is too small, while a long flat tail suggests the budget is being spent for nothing. It cannot tell you anything about overfitting — for that you would need a held-out split, which this MVP deliberately does not have.

Everything after it is in the original water-level unit: aggregate MAE/RMSE, the same metrics per lead hour, and the long-form preview of the first five forecast origins.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses)
plt.xlabel("Epoch")
plt.ylabel("Training MSE (scaled targets)")
plt.title(f"{CELL_TYPE.upper()} training loss")
plt.grid(alpha=0.3)
plt.show()

aggregate_metrics, per_horizon_metrics = metric_tables(
    test_targets, test_predictions, station_id=station_id
)
print(f"{CELL_TYPE.upper()} test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_origins, test_targets, test_predictions))